## Why Dataset Quality Matters

A machine learning model learns from the data we provide.

Problems such as corrupted images, incorrect labels, duplicate images,
or data leakage can negatively affect training and can produce misleading
evaluation results.

Therefore, dataset quality is checked before model training.

This script checks the animal image dataset for duplicate images
across the train, validation, and test splits.

Why is this necessary?
----------------------
Duplicate images between dataset splits can cause data leakage.

For example, if the same image exists in both the training and test
sets, the model may be evaluated on an image that it has already seen
during training. This can result in overly optimistic evaluation
metrics.

Detection Method
----------------
This script detects EXACT duplicate images using hashing.

For each image:
1. The image is opened using PIL.
2. It is converted to RGB to standardize the representation.
3. Its pixel data is extracted.
4. An MD5 hash is generated from the pixel data.
5. Images producing the same hash are considered exact duplicates.


In [1]:
from pathlib import Path
from PIL import Image
import hashlib

In [ ]:
DATA_DIR = Path("../data")

splits = ["train", "test", "test"]

def get_image_hash(image_path):
    with Image.open(image_path) as img:
        img = img.convert("RGB")
        return hashlib.md5(img.tobytes()).hexdigest()

hashes = {}

for split in splits:
    print(f"\nChecking {split}...")

    for class_path in (DATA_DIR / split).iterdir():
        if not class_path.is_dir():
            continue

        for image_path in class_path.iterdir():
            if image_path.is_file():
                try:
                    image_hash = get_image_hash(image_path)

                    if image_hash not in hashes:
                        hashes[image_hash] = []

                    hashes[image_hash].append(
                        (split, class_path.name, str(image_path))
                    )

                except Exception:
                    pass

duplicates = {
    h: files
    for h, files in hashes.items()
    if len(files) > 1
}

print("\n" + "=" * 50)
print("DUPLICATE CHECK")
print("=" * 50)

print("Unique images:", len(hashes))
print("Duplicate groups:", len(duplicates))

